# Function Tools

This notebook accompanies the [Agent Foundry](https://agent-foundry-pi.vercel.app) OpenAI Agents SDK roadmap.

You will learn how to create function tools with `@function_tool`, attach them to agents, and use Pydantic models as tool inputs.

## 1. Install Dependencies

In [ ]:
!pip install -q openai-agents

## 2. Set Up Your API Key

In [ ]:
import os
from getpass import getpass

os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

## 3. Basic Function Tool

The `@function_tool` decorator turns a plain Python function into a tool. The SDK generates the JSON schema from the function name, docstring, and type hints.

In [ ]:
from agents import Agent, Runner, function_tool


@function_tool
def get_weather(city: str) -> str:
    """Get the current weather for a given city."""
    return f"The weather in {city} is 72°F and sunny."


agent = Agent(
    name="Weather Bot",
    instructions="You help users check the weather. Use your tool to look up weather information.",
    tools=[get_weather],
)

result = Runner.run_sync(agent, "What's the weather in Tokyo?")
print(result.final_output)

## 4. Schema Generation from Type Hints

Optional parameters with defaults are reflected in the generated schema. The model sees `query` as required and `max_results` as optional.

In [ ]:
@function_tool
def search_products(query: str, max_results: int = 5) -> str:
    """Search the product catalog by keyword.

    Args:
        query: The search term to look for.
        max_results: Maximum number of results to return.
    """
    return f"Found {max_results} products matching '{query}'"


shop_agent = Agent(
    name="Shop Assistant",
    instructions="You help users find products.",
    tools=[search_products],
)

result = Runner.run_sync(shop_agent, "Find me some running shoes")
print(result.final_output)

## 5. Multiple Tools on One Agent

Pass multiple tools in the `tools` list. The agent decides which tool to call based on the user's request.

In [ ]:
@function_tool
def add(a: float, b: float) -> str:
    """Add two numbers together."""
    return str(a + b)


@function_tool
def multiply(a: float, b: float) -> str:
    """Multiply two numbers together."""
    return str(a * b)


calculator = Agent(
    name="Calculator",
    instructions="You are a calculator. Use your tools to perform math operations.",
    tools=[add, multiply],
)

result = Runner.run_sync(calculator, "What is 7 * 8 + 3?")
print(result.final_output)

## 6. Async Function Tools

Use `async def` for tools that perform I/O. Async tools require the async runner.

In [ ]:
@function_tool
async def slow_lookup(topic: str) -> str:
    """Simulate a slow database lookup."""
    import asyncio
    await asyncio.sleep(1)
    return f"Here is the data about {topic}: [simulated result]"


research_agent = Agent(
    name="Researcher",
    instructions="You help users research topics. Use your lookup tool.",
    tools=[slow_lookup],
)

result = await Runner.run(research_agent, "Look up quantum computing")
print(result.final_output)

## 7. Pydantic Models as Tool Inputs

For complex inputs, define a Pydantic model. The SDK generates nested JSON schema automatically.

In [ ]:
from pydantic import BaseModel


class FlightSearch(BaseModel):
    origin: str
    destination: str
    date: str
    max_price: float = 1000.0


@function_tool
def search_flights(search: FlightSearch) -> str:
    """Search for available flights matching the criteria."""
    return (
        f"Found 3 flights from {search.origin} to {search.destination} "
        f"on {search.date} under ${search.max_price}"
    )


travel_agent = Agent(
    name="Travel Agent",
    instructions="You help users find flights. Use your search tool.",
    tools=[search_flights],
)

result = Runner.run_sync(
    travel_agent, "Find flights from NYC to London on March 15th under $800"
)
print(result.final_output)

## Key Takeaways

- `@function_tool` turns any Python function into an agent tool with zero boilerplate
- The SDK generates the JSON schema from the function name, docstring, and type hints
- Both sync and async functions are supported
- Pass tools to an agent via `Agent(tools=[...])`
- Use Pydantic models for complex, validated tool inputs